# HW3: Frontier Lab Interview Prep

**CMSC 25750: Large Language Models** | Instructor: Chenhao Tan

This notebook is your submission file. Implement the functions in `src/distributed.py` and `src/attn_residuals.py`, then run all cells in order and write your answers in the designated markdown cells.

Unlike HW2, no `compile.py` or GPU is needed — both questions run on CPU. The `!pytest` cells below invoke the autograders directly.

---
**Questions**
- [Question 1: Distributed Training](#question1)
- [Question 2: Attention Residuals](#question2)


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import torch

print(f'torch  : {torch.__version__}')
print(f'device : {"cuda" if torch.cuda.is_available() else "cpu"} (CPU is fine for HW3)')


---
<a id='question1'></a>
## Question 1: Distributed Training (50 pts)

Implement `ring_allreduce` and `simulate_ddp_step` in `src/distributed.py`, then run the autograder below.

In [ ]:
# Autograder for Question 1
# Run this cell after implementing src/distributed.py
!{sys.executable} -m pytest tests/test_distributed.py -v 2>&1 | tail -30


### 1.3(a) — Communication cost (9 pts)

> You have $N = 8$ A100 GPUs connected by NVLink at 300 GB/s (bidirectional per link), training a 7B-parameter model in FP16 ($b = 2$ bytes/param).
>
> (i) How many bytes does each GPU send during ring allreduce?
>
> (ii) Estimate the allreduce's wall-clock time, i.e., the elapsed real-world time, assuming the operation is *bandwidth-bound* on a fully-utilized NVLink ring (ignore per-message latency and setup overhead, and treat the ring as a pipeline whose throughput is set by its slowest link). In this regime, wall-clock time = per-GPU bytes sent / link bandwidth.
>
> (iii) Suppose the forward + backward pass takes 1.5 seconds and communication runs *serially* after compute (i.e., no overlap between the two). What fraction of the total step time (compute + communication) is spent on communication?

**YOUR ANSWER:**

*(Numerical calculations.)*


### 1.3(b) — Scaling and design tradeoffs (12 pts)

> (i) Suppose the 8 GPUs are spread across 2 nodes connected by InfiniBand at 50 GB/s (instead of NVLink at 300 GB/s). Recompute the allreduce time. What fraction of step time is communication now? What does this tell you about the difference between *intra-node* and *inter-node* training?
>
> (ii) DDP overlaps communication with the backward pass by firing off allreduce for each *gradient bucket* (a group of parameters bundled together for one allreduce, ~25 MB by default) as soon as all gradients in that bucket are ready. Explain why this helps and what limits the improvement (i.e., when does overlap fail to fully hide communication?).

**YOUR ANSWER:**

*(Calculations + 2–3 sentences of explanation per part.)*


---
<a id='question2'></a>
## Question 2: Attention Residuals (50 pts)

Implement `prenorm_residual_layer`, `full_attn_res`, and `block_attn_res` in `src/attn_residuals.py`, then run the autograder below.

In [ ]:
# Autograder for Question 2
# Run this cell after implementing src/attn_residuals.py
!{sys.executable} -m pytest tests/test_attn_res.py -v 2>&1 | tail -40


### 2.4(a) — Cost of Full vs. Block AttnRes (8 pts)

> Consider a model with $L$ layers, hidden dim $d$, sequence length $T$, batch size $B$, and FP16 activations (2 bytes per element).
>
> (i) How many bytes of *additional* activation memory does Full AttnRes require at the deepest layer? What about Block AttnRes with $N$ blocks? Be precise about which tensors must be kept live and why.
>
> (ii) How many extra FLOPs per token does the depth-wise attention add at the deepest layer for Full AttnRes versus Block AttnRes? Plug in $L = 64$, $N = 8$, $T = 4096$, $d = 4096$ and compare both numbers to the per-layer self-attention cost $O(Td^2 + T^2d)$. Why is the AttnRes overhead negligible at this scale?

**YOUR ANSWER:**

*(Calculations with 2–3 sentences of explanation per part.)*


### 2.4(b) — PreNorm dilution (12 pts)

> Throughout this problem, write $f_i := f_i(\text{LN}(\mathbf{h}_i))$ as shorthand for the PreNorm sublayer output (so the input LayerNorm is implicit). Recall that a standard PreNorm Transformer produces $\mathbf{h}_l = \mathbf{h}_1 + \sum_{i=1}^{l-1} f_i$, whereas AttnRes produces $\mathbf{h}_l = \sum_{i=0}^{l-1} \alpha_{i \to l} \mathbf{v}_i$ with $\alpha_{i \to l} \ge 0$ and $\sum_i \alpha_{i \to l} = 1$.
>
> **(i) Norm growth under standard PreNorm.** We will show that $\|\mathbf{h}_l\|$ grows like $\sqrt{l}$, so a single layer's relative contribution shrinks like $1/\sqrt{l}$. Assume the layer outputs are approximately orthogonal in high dimension: $\mathbb{E}\langle f_i, f_j\rangle \approx 0$ for $i \ne j$, and each $\mathbb{E}\|f_i\|^2 = c^2$ for some $c > 0$.
>
> - **(a) Expand.** Using bilinearity of the inner product ($\|\mathbf{u}\|^2 = \langle \mathbf{u}, \mathbf{u}\rangle$), expand $\|\mathbf{h}_1 + \sum_{i=1}^{l-1} f_i\|^2$ into four term groups: $\|\mathbf{h}_1\|^2$, the $\mathbf{h}_1$–$f_i$ cross terms, the diagonal $\|f_i\|^2$ terms, and the $f_i$–$f_j$ cross terms ($i \ne j$). No probabilistic assumptions needed here.
> - **(b) Take expectations.** Apply the assumptions term-by-term to show $\mathbb{E}\|\mathbf{h}_l\|^2 \approx \|\mathbf{h}_1\|^2 + (l-1)c^2$.
> - **(c) Conclude.** Since $\|\mathbf{h}_l\| \sim c\sqrt{l}$ but a single layer's contribution has norm $\sim c$, the relative contribution $\|f_i\| / \|\mathbf{h}_l\|$ shrinks like $1/\sqrt{l}$. In one sentence, explain why this is undesirable: how does it make deep models harder to train?
>
> **(ii) Bounded norm under AttnRes.** Show that AttnRes, whose mixing weights satisfy $\sum_i \alpha_{i \to l} = 1$ and $\alpha_{i \to l} \ge 0$, keeps $\|\mathbf{h}_l\|$ bounded by $\max_i \|\mathbf{v}_i\|$ independent of depth. In one short sentence: where does the boundedness come from structurally?

**YOUR ANSWER:**

*(A few lines of math per sub-part, plus the two short explanations requested in (i)(c) and (ii).)*


---
## Submit

Export this notebook to PDF (File → Save and Export Notebook As → PDF) and upload to Gradescope. Do **not** submit `src/` files or a separate PDF — the notebook PDF is the single submission artifact.
